# 00 Replace-Oriented Dog Pose Indexing

This notebook is not meant to be a generic pose-analysis study. Its job is to build a **searchable, explainable, and directly usable** pose index for the downstream dog-replacement pipeline.

The labeling principle is simple: **every label should support replacement decisions**.

After the run, each image receives fields that are directly useful for replacement:

- `view_class`: side profile, frontal/rear view, or ambiguous view
- `pose_geometry_label`: geometry-only pose label after removing left/right direction, for example `horizontal|mid|extended`
- `visible_parts_signature`: which body parts are truly visible, for example `head+torso+front_legs+hind_legs+tail`
- `coverage_class`: full body, front half, rear half, head-only, or another partial view
- `obstruction_class`: mostly clear, occluded, or truncated by the frame
- `replace_mode`: suitable for `full_body`, `partial_body`, or should be `reject`
- `candidate_pool_key`: which downstream replacement pool this image should enter

This is more useful than a simple label such as `left|horizontal|mid|extended` because:

- left and right can be mirrored, so they should not be treated as a hard filter
- frontal images often do not provide stable body and leg geometry for full-body replacement
- partial or occluded dogs are still usable, but only within a compatible visible-part category


In [ ]:
%matplotlib inline

## What This Step Produces

The output of `00` is not a final image. It is a **pose index plus candidate-pool logic** that the later `01/02/03` notebooks can consume directly.

1. Run dog-pose inference over the full `data/Images/` dataset
2. Assign replacement-oriented labels to every image
3. Save the index as JSON and CSV
4. Build candidate-pool listings for later coarse filtering
5. Provide visualizations that explain why two dogs are compatible or incompatible for replacement


In [ ]:
import csv
import json
import math
import os
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils import patches as ultralytics_patches


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "Images").exists() and (candidate / "notebooks").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


def resolve_pose_model_path(project_root):
    candidates = [
        Path(os.environ["DOG_POSE_MODEL_PATH"]) if os.environ.get("DOG_POSE_MODEL_PATH") else None,
        project_root / "models" / "dog_pose_best.pt",
        project_root / "dog_pose_best.pt",
        project_root / "data" / "outputs" / "dog_pose_training" / "dog_pose_bootstrap" / "weights" / "best.pt",
    ]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate
    return project_root / "models" / "dog_pose_best.pt"


if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a local CUDA GPU, but no CUDA device is available.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_grad_enabled(False)

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "Images"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs" / "pose_index"
POOL_DIR = OUTPUT_DIR / "candidate_pools"
MODE_DIR = OUTPUT_DIR / "replace_modes"
VIEW_DIR = OUTPUT_DIR / "view_classes"
CURRENT_STAGE_DIR = OUTPUT_DIR / "current_stage_side_profiles"
CURRENT_STAGE_POOL_DIR = CURRENT_STAGE_DIR / "candidate_pools"
REPLACEABLE_GROUP_DIR = PROJECT_ROOT / "data" / "replacbale_group"
for folder in [OUTPUT_DIR, POOL_DIR, MODE_DIR, VIEW_DIR, CURRENT_STAGE_DIR, CURRENT_STAGE_POOL_DIR, REPLACEABLE_GROUP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

POSE_MODEL_PATH = resolve_pose_model_path(PROJECT_ROOT)
IMG_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
POSE_CONF = 0.15
POSE_IMGSZ = 960
KEYPOINT_CONF = 0.20

DOG_KEYPOINT_NAMES = [
    "front_left_paw",
    "front_left_knee",
    "front_left_elbow",
    "rear_left_paw",
    "rear_left_knee",
    "rear_left_elbow",
    "front_right_paw",
    "front_right_knee",
    "front_right_elbow",
    "rear_right_paw",
    "rear_right_knee",
    "rear_right_elbow",
    "tail_start",
    "tail_end",
    "left_ear_base",
    "right_ear_base",
    "nose",
    "chin",
    "left_ear_tip",
    "right_ear_tip",
    "left_eye",
    "right_eye",
    "withers",
    "throat",
]
NAME_TO_INDEX = {name: idx for idx, name in enumerate(DOG_KEYPOINT_NAMES)}

PART_GROUPS = {
    "head": ["nose", "chin", "left_eye", "right_eye", "left_ear_base", "right_ear_base", "left_ear_tip", "right_ear_tip"],
    "torso": ["withers", "throat", "tail_start"],
    "front_legs": ["front_left_paw", "front_left_knee", "front_left_elbow", "front_right_paw", "front_right_knee", "front_right_elbow"],
    "hind_legs": ["rear_left_paw", "rear_left_knee", "rear_left_elbow", "rear_right_paw", "rear_right_knee", "rear_right_elbow"],
    "tail": ["tail_start", "tail_end"],
}

Image.open = ultralytics_patches._image_open
ultralytics_patches.Image.open = ultralytics_patches._image_open
ultralytics_patches._pil_plugins_registered = False
plt.rcParams["axes.grid"] = False
plt.rcParams["figure.figsize"] = (12, 8)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")
if not POSE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Dog pose checkpoint not found at {POSE_MODEL_PATH}. Put the model there or set DOG_POSE_MODEL_PATH."
    )

pose_model = YOLO(str(POSE_MODEL_PATH))
model_kpt_shape = getattr(getattr(pose_model, "model", None), "kpt_shape", None)
if model_kpt_shape is not None and int(model_kpt_shape[0]) != 24:
    raise RuntimeError(f"Expected a 24-keypoint dog pose checkpoint, but got {model_kpt_shape}.")

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Pose model:", POSE_MODEL_PATH)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Model kpt_shape:", model_kpt_shape)


## Collect Images To Index

This step scans `data/Images/`. The notebook will later identify one primary dog per image and assign replacement-oriented pose labels.


In [ ]:
image_paths = sorted([p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in IMG_SUFFIXES])
if not image_paths:
    raise RuntimeError(f"No images found under {DATA_ROOT}")

breed_counts = Counter(path.parent.name for path in image_paths)
print(f"Found {len(image_paths)} images across {len(breed_counts)} folders.")
print("Top folders by image count:")
for breed, count in breed_counts.most_common(20):
    print(f" - {breed}: {count}")


## Label Logic: Designed For Replacement

The goal here is not "the more detailed the taxonomy, the better." The goal is "can these labels become practical rules in the replacement pipeline?"

The most important questions are:

- Is this a side-profile image? If not, full-body replacement is usually much less stable.
- Which body parts are actually visible?
- Is this image suitable for full-body replacement or only partial-body replacement?
- Which other images belong in the same candidate pool?

Important: `left` and `right` are no longer hard filters. For side-profile dogs, the pose is canonicalized before similarity comparison so that mirroring remains a valid downstream operation.


In [ ]:
def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def rel_path(path):
    return str(path.resolve().relative_to(PROJECT_ROOT)).replace("\\", "/")


def clip_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = int(max(0, min(width - 1, math.floor(x1))))
    y1 = int(max(0, min(height - 1, math.floor(y1))))
    x2 = int(max(x1 + 1, min(width, math.ceil(x2))))
    y2 = int(max(y1 + 1, min(height, math.ceil(y2))))
    return [x1, y1, x2, y2]


def get_keypoint_arrays(result, instance_idx):
    xy = result.keypoints.xy[instance_idx].detach().cpu().numpy().astype(np.float32)
    conf_tensor = getattr(result.keypoints, "conf", None)
    if conf_tensor is not None:
        conf = conf_tensor[instance_idx].detach().cpu().numpy().astype(np.float32)
    else:
        data = result.keypoints.data[instance_idx].detach().cpu().numpy().astype(np.float32)
        conf = data[:, 2] if data.shape[-1] >= 3 else np.ones(len(xy), dtype=np.float32)
    return xy, conf



def named_point(points, confs, name, threshold=KEYPOINT_CONF):
    idx = NAME_TO_INDEX[name]
    if confs[idx] >= threshold:
        return points[idx]
    return None


def count_visible(points, confs, names, threshold=KEYPOINT_CONF):
    return sum(1 for name in names if named_point(points, confs, name, threshold) is not None)


def mean_visible_point(points, confs, names, threshold=KEYPOINT_CONF):
    pts = [named_point(points, confs, name, threshold) for name in names]
    pts = [pt for pt in pts if pt is not None]
    if not pts:
        return None
    return np.mean(np.stack(pts, axis=0), axis=0)


def orientation_label(points, confs):
    nose = named_point(points, confs, "nose")
    rear_center = mean_visible_point(
        points,
        confs,
        ["tail_start", "tail_end", "rear_left_elbow", "rear_right_elbow", "rear_left_knee", "rear_right_knee", "rear_left_paw", "rear_right_paw"],
    )
    if nose is not None and rear_center is not None:
        return "right" if nose[0] > rear_center[0] else "left"
    return "unknown"


def classify_view(points, confs):
    orientation = orientation_label(points, confs)
    head_visible = count_visible(points, confs, PART_GROUPS["head"]) >= 2
    front_visible = count_visible(points, confs, PART_GROUPS["front_legs"]) >= 3
    hind_visible = count_visible(points, confs, PART_GROUPS["hind_legs"]) >= 2
    tail_visible = count_visible(points, confs, PART_GROUPS["tail"]) >= 1
    if orientation in {"left", "right"} and head_visible and (front_visible or hind_visible or tail_visible):
        return "side_profile"
    if orientation == "unknown" and head_visible and front_visible and not hind_visible:
        return "frontal_or_rear"
    if orientation == "unknown" and head_visible and not tail_visible and count_visible(points, confs, PART_GROUPS["head"]) >= 4:
        return "frontal_or_rear"
    return "ambiguous"


def visible_parts(points, confs):
    head_count = count_visible(points, confs, PART_GROUPS["head"])
    front_count = count_visible(points, confs, PART_GROUPS["front_legs"])
    hind_count = count_visible(points, confs, PART_GROUPS["hind_legs"])
    tail_count = count_visible(points, confs, PART_GROUPS["tail"])
    torso_count = count_visible(points, confs, PART_GROUPS["torso"])

    parts = []
    if head_count >= 2:
        parts.append("head")

    inferred_torso = torso_count >= 1 or ((front_count >= 2 and hind_count >= 2) or (head_count >= 2 and tail_count >= 1 and (front_count >= 2 or hind_count >= 2)))
    if inferred_torso:
        parts.append("torso")

    if front_count >= 2:
        parts.append("front_legs")
    if hind_count >= 2:
        parts.append("hind_legs")
    if tail_count >= 1:
        parts.append("tail")
    return parts


def visible_parts_signature(parts):
    return "+".join(parts) if parts else "none"


def coverage_class(parts):
    part_set = set(parts)
    has_head = "head" in part_set
    has_torso = "torso" in part_set
    has_front = "front_legs" in part_set
    has_hind = "hind_legs" in part_set
    has_tail = "tail" in part_set
    if has_head and has_front and has_hind:
        return "full_body"
    if has_head and has_front and not has_hind:
        return "front_half"
    if has_hind and (has_tail or has_torso) and not has_head:
        return "rear_half"
    if has_head and not (has_torso or has_front or has_hind):
        return "head_only"
    if has_head and has_torso and not (has_front or has_hind):
        return "upper_body"
    if has_front and has_hind and not has_head:
        return "lower_body"
    if has_torso and not (has_head or has_front or has_hind):
        return "torso_only"
    return "misc_partial"


def body_angle_bucket(points, confs):
    nose = named_point(points, confs, "nose")
    tail_start = named_point(points, confs, "tail_start")
    if nose is None or tail_start is None:
        return "unknown", None
    dy = float(nose[1] - tail_start[1])
    dx = float(nose[0] - tail_start[0])
    angle = abs(math.degrees(math.atan2(dy, dx)))
    if angle > 90.0:
        angle = 180.0 - angle
    if angle < 25.0:
        return "horizontal", angle
    if angle < 60.0:
        return "diagonal", angle
    return "vertical", angle


def posture_bucket(points, confs, bbox):
    x1, y1, x2, y2 = bbox
    w = max(1.0, x2 - x1)
    h = max(1.0, y2 - y1)
    aspect = h / w
    withers = named_point(points, confs, "withers")
    paw_names = ["front_left_paw", "front_right_paw", "rear_left_paw", "rear_right_paw"]
    paws = [named_point(points, confs, name) for name in paw_names]
    paws = [pt for pt in paws if pt is not None]
    paw_gap = None
    if withers is not None and paws:
        paw_gap = (float(np.mean([pt[1] for pt in paws])) - float(withers[1])) / h
    if aspect < 0.55 or (paw_gap is not None and paw_gap < 0.18):
        return "low"
    if aspect > 1.05 and (paw_gap is None or paw_gap > 0.30):
        return "tall"
    return "mid"


def spread_bucket(points, confs, bbox):
    x1, y1, x2, y2 = bbox
    w = max(1.0, x2 - x1)
    paw_names = ["front_left_paw", "front_right_paw", "rear_left_paw", "rear_right_paw"]
    paws = [named_point(points, confs, name) for name in paw_names]
    paws = [pt for pt in paws if pt is not None]
    if len(paws) < 2:
        return "unknown"
    spread = (max(float(pt[0]) for pt in paws) - min(float(pt[0]) for pt in paws)) / w
    if spread > 0.55:
        return "extended"
    if spread < 0.28:
        return "compact"
    return "balanced"


def frame_fit_bucket(bbox, image_shape):
    x1, y1, x2, y2 = bbox
    h, w = image_shape[:2]
    x_tol = max(6.0, w * 0.02)
    y_tol = max(6.0, h * 0.02)
    touches = 0
    if x1 <= x_tol:
        touches += 1
    if y1 <= y_tol:
        touches += 1
    if x2 >= w - x_tol:
        touches += 1
    if y2 >= h - y_tol:
        touches += 1
    if touches == 0:
        return "in_frame", touches
    if touches == 1:
        return "touching_edge", touches
    return "cut_off", touches



def obstruction_class(view_class, coverage, frame_fit, visible_count):
    if frame_fit == "cut_off":
        return "truncated_by_frame"
    if view_class == "frontal_or_rear":
        return "front_view_limited"
    if coverage == "full_body" and visible_count >= 14:
        return "clear"
    if visible_count >= 8:
        return "partially_occluded"
    return "heavily_occluded"


def replace_mode(view_class, coverage, frame_fit, visible_count):
    if frame_fit == "cut_off" and visible_count < 10:
        return "reject"
    if view_class == "side_profile":
        if coverage == "full_body" and frame_fit != "cut_off" and visible_count >= 12:
            return "full_body"
        if coverage != "misc_partial" and visible_count >= 8:
            return "partial_body"
        return "reject"
    if view_class in {"frontal_or_rear", "ambiguous"}:
        if coverage in {"head_only", "front_half", "upper_body", "misc_partial"} and visible_count >= 8 and frame_fit != "cut_off":
            return "partial_body"
        return "reject"
    return "reject"


def current_pipeline_gate(view_class, coverage, part_signature, frame_fit, obstruction, visible_count, bbox_area_ratio, dog_instance_count):
    if dog_instance_count > 1:
        return False, "multiple_dogs_present"
    if view_class != "side_profile":
        return False, "not_side_profile"
    if coverage != "full_body":
        return False, "not_full_body"
    if part_signature != "head+torso+front_legs+hind_legs+tail":
        return False, "missing_required_parts"
    if frame_fit == "cut_off":
        return False, "truncated_by_frame"
    if obstruction == "heavily_occluded":
        return False, "heavily_occluded"
    if visible_count < 12:
        return False, "too_few_keypoints"
    if bbox_area_ratio > 0.45:
        return False, "dog_too_large_in_frame"
    return True, "pass"


def canonical_normalized_keypoints(points, confs, bbox, view_class, orientation):
    x1, y1, x2, y2 = bbox
    visible = confs >= KEYPOINT_CONF
    if int(visible.sum()) == 0:
        return [[None, None] for _ in DOG_KEYPOINT_NAMES]
    tail_start = named_point(points, confs, "tail_start")
    withers = named_point(points, confs, "withers")
    nose = named_point(points, confs, "nose")
    throat = named_point(points, confs, "throat")
    if tail_start is not None and withers is not None:
        center = (tail_start + withers) / 2.0
    else:
        center = points[visible].mean(axis=0)
    scale = None
    if tail_start is not None and nose is not None:
        scale = float(np.linalg.norm(nose - tail_start))
    elif withers is not None and throat is not None:
        scale = float(np.linalg.norm(withers - throat) * 4.0)
    if scale is None or scale < 1e-3:
        scale = float(np.linalg.norm(np.array([x2 - x1, y2 - y1], dtype=np.float32)))
    scale = max(scale, 1.0)
    normalized = (points - center[None, :]) / scale
    normalized[~visible] = np.nan
    if view_class == "side_profile" and orientation == "right":
        normalized[:, 0] *= -1.0
    output = []
    for pt in normalized:
        if np.isfinite(pt).all():
            output.append([round(float(pt[0]), 6), round(float(pt[1]), 6)])
        else:
            output.append([None, None])
    return output


def choose_best_instance(result, min_visible=6):
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return None
    candidates = []
    for idx, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].detach().cpu().numpy().astype(np.float32)
        area = max(1.0, (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1]))
        score = float(box.conf.item())
        visible = float((get_keypoint_arrays(result, idx)[1] >= KEYPOINT_CONF).sum())
        if visible < min_visible:
            continue
        candidates.append((idx, score * math.sqrt(area) * (1.0 + visible / 24.0)))
    if not candidates:
        return None
    return max(candidates, key=lambda item: item[1])[0]


def count_valid_dog_instances(result, min_visible=6):
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return 0
    count = 0
    for idx, box in enumerate(result.boxes):
        score = float(box.conf.item())
        visible = int((get_keypoint_arrays(result, idx)[1] >= KEYPOINT_CONF).sum())
        if score >= POSE_CONF and visible >= min_visible:
            count += 1
    return count


def infer_pose_record(image_path):
    image = read_rgb(image_path)
    results = pose_model.predict(image, conf=POSE_CONF, imgsz=POSE_IMGSZ, device=0, verbose=False)
    result = results[0]
    best_idx = choose_best_instance(result)
    dog_instance_count = count_valid_dog_instances(result)
    base = {
        "relative_path": rel_path(image_path),
        "breed_folder": image_path.parent.name,
        "width": int(image.shape[1]),
        "height": int(image.shape[0]),
        "dog_instance_count": int(dog_instance_count),
    }
    if best_idx is None:
        base.update(
            {
                "status": "no_pose",
                "bbox_xyxy": None,
                "detection_conf": None,
                "visible_keypoints": 0,
                "original_orientation": "unknown",
                "view_class": "ambiguous",
                "pose_geometry_label": "unknown|unknown|unknown",
                "visible_parts_signature": "none",
                "coverage_class": "misc_partial",
                "obstruction_class": "heavily_occluded",
                "replace_mode": "reject",
                "candidate_pool_key": "reject|ambiguous|unknown|misc_partial|none",
                "frame_fit_bucket": "unknown",
                "body_angle_deg": None,
                "keypoints_xy": None,
                "keypoints_conf": None,
                "normalized_keypoints": None,
                "dog_instance_count": int(dog_instance_count),
                "current_pipeline_pass": False,
                "current_pipeline_reason": "no_pose",
                "current_candidate_pool_key": "reject|no_pose",
            }
        )
        return base

    xyxy = result.boxes[best_idx].xyxy[0].detach().cpu().numpy().astype(np.float32)
    bbox = clip_bbox(xyxy.tolist(), image.shape[1], image.shape[0])
    keypoints_xy, keypoints_conf = get_keypoint_arrays(result, best_idx)
    visible_count = int((keypoints_conf >= KEYPOINT_CONF).sum())
    orientation = orientation_label(keypoints_xy, keypoints_conf)
    view = classify_view(keypoints_xy, keypoints_conf)
    parts = visible_parts(keypoints_xy, keypoints_conf)
    part_signature = visible_parts_signature(parts)
    coverage = coverage_class(parts)
    angle_bucket, body_angle = body_angle_bucket(keypoints_xy, keypoints_conf)
    posture = posture_bucket(keypoints_xy, keypoints_conf, bbox)
    spread = spread_bucket(keypoints_xy, keypoints_conf, bbox)
    geometry = "|".join([angle_bucket, posture, spread])
    bbox_area_ratio = round(float(((bbox[2] - bbox[0]) * (bbox[3] - bbox[1])) / max(1, image.shape[0] * image.shape[1])), 6)
    frame_fit, edge_touches = frame_fit_bucket(bbox, image.shape)
    obstruction = obstruction_class(view, coverage, frame_fit, visible_count)
    mode = replace_mode(view, coverage, frame_fit, visible_count)
    current_pipeline_pass, current_pipeline_reason = current_pipeline_gate(
        view,
        coverage,
        part_signature,
        frame_fit,
        obstruction,
        visible_count,
        bbox_area_ratio,
        dog_instance_count,
    )
    candidate_pool_key = "|".join([mode, view, geometry, coverage, part_signature])
    current_candidate_pool_key = "|".join([view, geometry, coverage, part_signature]) if current_pipeline_pass else f"reject|{current_pipeline_reason}"
    normalized = canonical_normalized_keypoints(keypoints_xy, keypoints_conf, bbox, view, orientation)

    base.update(
        {
            "status": "ok",
            "bbox_xyxy": [int(v) for v in bbox],
            "bbox_area_ratio": bbox_area_ratio,
            "detection_conf": round(float(result.boxes[best_idx].conf.item()), 6),
            "dog_instance_count": int(dog_instance_count),
            "visible_keypoints": visible_count,
            "visible_keypoint_names": [name for name, conf in zip(DOG_KEYPOINT_NAMES, keypoints_conf.tolist()) if conf >= KEYPOINT_CONF],
            "original_orientation": orientation,
            "view_class": view,
            "pose_geometry_label": geometry,
            "visible_parts_signature": part_signature,
            "coverage_class": coverage,
            "obstruction_class": obstruction,
            "replace_mode": mode,
            "candidate_pool_key": candidate_pool_key,
            "current_pipeline_pass": bool(current_pipeline_pass),
            "current_pipeline_reason": current_pipeline_reason,
            "current_candidate_pool_key": current_candidate_pool_key,
            "frame_fit_bucket": frame_fit,
            "frame_edge_touch_count": int(edge_touches),
            "body_angle_deg": None if body_angle is None else round(float(body_angle), 4),
            "keypoints_xy": [[round(float(x), 3), round(float(y), 3)] for x, y in keypoints_xy.tolist()],
            "keypoints_conf": [round(float(v), 6) for v in keypoints_conf.tolist()],
            "normalized_keypoints": normalized,
        }
    )
    return base


## Run Full-Dataset Indexing And Save Results

In addition to the master table, this step also exports:

- one text listing per candidate pool

When we do real replacement later, we first retrieve candidates from the matching `candidate_pool_key`, and only then do a finer ranking using keypoint similarity.


In [ ]:
def sanitize_group_name(label):
    safe_name = label
    for bad_char, replacement in [
        ("|", "__"),
        ("/", "-"),
        ("\\", "-"),
        (":", "-"),
        ("*", "-"),
        ("?", ""),
        ('"', ""),
        ("<", "("),
        (">", ")"),
    ]:
        safe_name = safe_name.replace(bad_char, replacement)
    return safe_name


def write_group_summary(group_map, summary_path, group_dir, label_field):
    rows = []
    for label, paths in sorted(group_map.items(), key=lambda item: (-len(item[1]), item[0])):
        safe_name = sanitize_group_name(label)
        manifest_path = group_dir / f"{safe_name}.txt"
        with open(manifest_path, "w", encoding="utf-8") as f:
            for item in paths:
                f.write(item + "\n")
        rows.append({label_field: label, "count": len(paths), "group_file": rel_path(manifest_path)})
    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[label_field, "count", "group_file"])
        writer.writeheader()
        writer.writerows(rows)
    return rows


def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def materialize_group_images(group_map, target_root):
    if target_root.exists():
        shutil.rmtree(target_root)
    target_root.mkdir(parents=True, exist_ok=True)
    rows = []
    for label, paths in sorted(group_map.items(), key=lambda item: (-len(item[1]), item[0])):
        safe_name = sanitize_group_name(label)
        group_dir = target_root / safe_name
        group_dir.mkdir(parents=True, exist_ok=True)
        manifest_path = group_dir / "_files.txt"
        linked_files = []
        for relative_path in paths:
            src = PROJECT_ROOT / relative_path
            if not src.exists():
                continue
            breed_name = src.parent.name
            dst_name = f"{breed_name}__{src.name}"
            dst = group_dir / dst_name
            link_or_copy(src, dst)
            linked_files.append(dst_name)
        with open(manifest_path, "w", encoding="utf-8") as f:
            for name in linked_files:
                f.write(name + "\n")
        rows.append(
            {
                "group_label": label,
                "count": len(linked_files),
                "folder_path": rel_path(group_dir),
                "manifest_path": rel_path(manifest_path),
            }
        )
    summary_path = target_root / "group_folders_summary.csv"
    with open(summary_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["group_label", "count", "folder_path", "manifest_path"])
        writer.writeheader()
        writer.writerows(rows)
    return rows, summary_path


records = []
errors = []
for idx, image_path in enumerate(image_paths, start=1):
    if idx == 1 or idx % 250 == 0 or idx == len(image_paths):
        print(f"[{idx}/{len(image_paths)}] {image_path.name}")
    try:
        records.append(infer_pose_record(image_path))
    except Exception as exc:
        errors.append({"relative_path": rel_path(image_path), "error": f"{type(exc).__name__}: {exc}"})

ok_records = [record for record in records if record["status"] == "ok"]
current_stage_records = [record for record in ok_records if record.get("current_pipeline_pass")]

pool_groups = defaultdict(list)
mode_groups = defaultdict(list)
view_groups = defaultdict(list)
current_stage_pool_groups = defaultdict(list)
current_stage_pass_groups = defaultdict(list)
replaceable_export_groups = defaultdict(list)
for record in records:
    if record["status"] == "ok":
        pool_groups[record["candidate_pool_key"]].append(record["relative_path"])
        mode_groups[record["replace_mode"]].append(record["relative_path"])
        view_groups[record["view_class"]].append(record["relative_path"])
    pass_key = "pass" if record.get("current_pipeline_pass") else f"reject::{record.get('current_pipeline_reason', 'unknown')}"
    current_stage_pass_groups[pass_key].append(record["relative_path"])
    if record.get("current_pipeline_pass"):
        current_stage_pool_groups[record["current_candidate_pool_key"]].append(record["relative_path"])
        replaceable_export_groups[record["current_candidate_pool_key"]].append(record["relative_path"])
    else:
        replaceable_export_groups["reject_nonreplaceable"].append(record["relative_path"])

json_path = OUTPUT_DIR / "dog_pose_index.json"
csv_path = OUTPUT_DIR / "dog_pose_index.csv"
pool_summary_path = OUTPUT_DIR / "candidate_pool_summary.csv"
mode_summary_path = OUTPUT_DIR / "replace_mode_summary.csv"
view_summary_path = OUTPUT_DIR / "view_class_summary.csv"
current_stage_summary_path = CURRENT_STAGE_DIR / "current_stage_pool_summary.csv"
current_stage_pass_summary_path = CURRENT_STAGE_DIR / "current_stage_pass_summary.csv"
current_stage_json_path = CURRENT_STAGE_DIR / "current_stage_side_profiles.json"
current_stage_csv_path = CURRENT_STAGE_DIR / "current_stage_side_profiles.csv"
errors_path = OUTPUT_DIR / "pose_index_errors.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

csv_fields = [
    "relative_path",
    "breed_folder",
    "status",
    "width",
    "height",
    "detection_conf",
    "bbox_area_ratio",
    "dog_instance_count",
    "visible_keypoints",
    "original_orientation",
    "view_class",
    "pose_geometry_label",
    "visible_parts_signature",
    "coverage_class",
    "obstruction_class",
    "replace_mode",
    "candidate_pool_key",
    "current_pipeline_pass",
    "current_pipeline_reason",
    "current_candidate_pool_key",
    "frame_fit_bucket",
    "frame_edge_touch_count",
    "body_angle_deg",
]
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for record in records:
        writer.writerow({field: record.get(field) for field in csv_fields})

with open(current_stage_json_path, "w", encoding="utf-8") as f:
    json.dump(current_stage_records, f, ensure_ascii=False, indent=2)
with open(current_stage_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for record in current_stage_records:
        writer.writerow({field: record.get(field) for field in csv_fields})

pool_rows = write_group_summary(pool_groups, pool_summary_path, POOL_DIR, "candidate_pool_key")
mode_rows = write_group_summary(mode_groups, mode_summary_path, MODE_DIR, "replace_mode")
view_rows = write_group_summary(view_groups, view_summary_path, VIEW_DIR, "view_class")
current_stage_rows = write_group_summary(
    current_stage_pool_groups,
    current_stage_summary_path,
    CURRENT_STAGE_POOL_DIR,
    "current_candidate_pool_key",
)
current_stage_pass_rows = write_group_summary(
    current_stage_pass_groups,
    current_stage_pass_summary_path,
    CURRENT_STAGE_DIR,
    "current_stage_pass_key",
)
replaceable_group_rows, replaceable_group_summary_path = materialize_group_images(
    replaceable_export_groups,
    REPLACEABLE_GROUP_DIR,
)

with open(errors_path, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"Saved {len(records)} records to {json_path}")
print(f"Successful pose detections: {len(ok_records)}")
print(f"Current-stage side-profile candidates: {len(current_stage_records)}")
print(f"Inference errors: {len(errors)}")
print(f"Candidate pools: {len(pool_rows)}")
print(f"Replace modes: {len(mode_rows)}")
print(f"View classes: {len(view_rows)}")
print(f"Current-stage pools: {len(current_stage_rows)}")
print(f"Current-stage pass groups: {len(current_stage_pass_rows)}")
print(f"Replaceable-group folders: {len(replaceable_group_rows)}")
print(f"Replaceable-group summary: {replaceable_group_summary_path}")


## Global View: Which Images Are Actually Replaceable?

This view is not only about pose. It helps answer dataset-level questions such as:

- How many images are good `full_body` candidates?
- How many are only suitable for `partial_body` replacement?
- What are the main reasons for `reject`?
- How strongly is the dataset affected by frontal or ambiguous views?


In [ ]:
replace_mode_counts = Counter(record["replace_mode"] for record in ok_records)
view_counts = Counter(record["view_class"] for record in ok_records)
coverage_counts = Counter(record["coverage_class"] for record in ok_records)
current_stage_counts = Counter("pass" if record.get("current_pipeline_pass") else "reject" for record in records)
current_stage_reason_counts = Counter(
    record.get("current_pipeline_reason", "unknown")
    for record in records
    if not record.get("current_pipeline_pass")
)

print("Replace mode distribution:")
for label, count in replace_mode_counts.most_common():
    print(f" - {label}: {count}")

print("\nView class distribution:")
for label, count in view_counts.most_common():
    print(f" - {label}: {count}")

print("\nCoverage distribution:")
for label, count in coverage_counts.most_common(10):
    print(f" - {label}: {count}")

print("\nCurrent-stage pass distribution:")
for label, count in current_stage_counts.most_common():
    print(f" - {label}: {count}")

print("\nCurrent-stage reject reasons:")
for label, count in current_stage_reason_counts.most_common(10):
    print(f" - {label}: {count}")

print("\nReplaceable groups exported to:")
print(REPLACEABLE_GROUP_DIR)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
chart_items = [
    (axes[0, 0], "Replace mode", replace_mode_counts),
    (axes[0, 1], "View class", view_counts),
    (axes[1, 0], "Coverage class", coverage_counts),
    (axes[1, 1], "Current-stage pass", current_stage_counts),
]
for ax, title, counts in chart_items:
    labels = list(counts.keys())
    values = list(counts.values())
    ax.bar(range(len(labels)), values)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_title(title)
    ax.set_ylabel("Image count")
plt.tight_layout()
plt.show()


## Single-Image Explanation, Candidate-Pool Browsing, And Nearest Neighbors

These helpers are here so that you can explain the indexing logic directly from the notebook:

- why one image becomes `full_body`, `partial_body`, or `reject`
- which candidate pool that image should use
- which dogs inside that pool are the closest pose matches


In [ ]:
RECORDS_BY_PATH = {record["relative_path"]: record for record in ok_records}
CURRENT_STAGE_RECORDS = [record for record in ok_records if record.get("current_pipeline_pass")]

DOG_SKELETON_EDGES = [
    ("nose", "chin"),
    ("nose", "left_eye"),
    ("nose", "right_eye"),
    ("left_eye", "left_ear_base"),
    ("right_eye", "right_ear_base"),
    ("left_ear_base", "left_ear_tip"),
    ("right_ear_base", "right_ear_tip"),
    ("throat", "nose"),
    ("throat", "withers"),
    ("withers", "tail_start"),
    ("tail_start", "tail_end"),
    ("withers", "front_left_elbow"),
    ("front_left_elbow", "front_left_knee"),
    ("front_left_knee", "front_left_paw"),
    ("withers", "front_right_elbow"),
    ("front_right_elbow", "front_right_knee"),
    ("front_right_knee", "front_right_paw"),
    ("tail_start", "rear_left_elbow"),
    ("rear_left_elbow", "rear_left_knee"),
    ("rear_left_knee", "rear_left_paw"),
    ("tail_start", "rear_right_elbow"),
    ("rear_right_elbow", "rear_right_knee"),
    ("rear_right_knee", "rear_right_paw"),
]


def draw_pose_record(relative_path, show_labels=False, figsize=(7, 7)):
    record = RECORDS_BY_PATH[relative_path]
    image = read_rgb(PROJECT_ROOT / relative_path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    if record.get("bbox_xyxy") is not None:
        x1, y1, x2, y2 = record["bbox_xyxy"]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
    point_map = {
        name: pt
        for name, pt, conf in zip(DOG_KEYPOINT_NAMES, record.get("keypoints_xy") or [], record.get("keypoints_conf") or [])
        if conf >= KEYPOINT_CONF
    }
    for start_name, end_name in DOG_SKELETON_EDGES:
        start = point_map.get(start_name)
        end = point_map.get(end_name)
        if start is None or end is None:
            continue
        ax.plot([start[0], end[0]], [start[1], end[1]], color="cyan", linewidth=2)
    for name, pt in point_map.items():
        ax.scatter(pt[0], pt[1], s=24, color="yellow", edgecolors="black", linewidths=0.5)
        if show_labels:
            ax.text(pt[0] + 3, pt[1] + 3, name, fontsize=7, color="white", bbox=dict(facecolor="black", alpha=0.55, pad=1))
    ax.set_title(
        f"{Path(relative_path).name}\n"
        f"{record['replace_mode']} | {record['view_class']} | {record['pose_geometry_label']}\n"
        f"dogs={record.get('dog_instance_count')} | current_stage={record.get('current_pipeline_pass')} | {record['visible_parts_signature']}"
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def print_record_summary(relative_path):
    record = RECORDS_BY_PATH[relative_path]
    fields = [
        "relative_path",
        "breed_folder",
        "replace_mode",
        "view_class",
        "pose_geometry_label",
        "visible_parts_signature",
        "coverage_class",
        "obstruction_class",
        "candidate_pool_key",
        "dog_instance_count",
        "current_pipeline_pass",
        "current_pipeline_reason",
        "current_candidate_pool_key",
        "original_orientation",
        "visible_keypoints",
        "frame_fit_bucket",
        "bbox_area_ratio",
    ]
    for field in fields:
        print(f"{field}: {record.get(field)}")


def records_in_group(group_value, group_field="candidate_pool_key"):
    source_records = CURRENT_STAGE_RECORDS if group_field == "current_candidate_pool_key" else ok_records
    return [record for record in source_records if record.get(group_field) == group_value]


def show_group_samples(group_value, group_field="candidate_pool_key", max_images=12, cols=4):
    records = records_in_group(group_value, group_field=group_field)[:max_images]
    if not records:
        print(f"No images found for {group_field}={group_value}")
        return
    rows = int(math.ceil(len(records) / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))
    for idx, record in enumerate(records, start=1):
        plt.subplot(rows, cols, idx)
        plt.imshow(read_rgb(PROJECT_ROOT / record["relative_path"]))
        plt.title(
            f"{Path(record['relative_path']).name}\n{record['replace_mode']} | {record['coverage_class']}",
            fontsize=8,
        )
        plt.axis("off")
    plt.suptitle(f"{group_field} = {group_value}")
    plt.tight_layout()
    plt.show()


def normalized_array_from_record(record):
    return np.array(record["normalized_keypoints"], dtype=np.float32)


def keypoint_distance(record_a, record_b, min_shared=8):
    a = normalized_array_from_record(record_a)
    b = normalized_array_from_record(record_b)
    shared = np.isfinite(a[:, 0]) & np.isfinite(a[:, 1]) & np.isfinite(b[:, 0]) & np.isfinite(b[:, 1])
    shared_count = int(shared.sum())
    if shared_count < min_shared:
        return math.inf, shared_count
    diff = a[shared] - b[shared]
    dist = float(np.sqrt((diff ** 2).sum(axis=1).mean()))
    return dist, shared_count


def visibility_overlap_penalty(record_a, record_b):
    conf_a = np.array(record_a["keypoints_conf"], dtype=np.float32)
    conf_b = np.array(record_b["keypoints_conf"], dtype=np.float32)
    vis_a = conf_a >= KEYPOINT_CONF
    vis_b = conf_b >= KEYPOINT_CONF
    union = vis_a | vis_b
    if int(union.sum()) == 0:
        return 1.0, 0.0
    overlap = int((vis_a & vis_b).sum()) / float(union.sum())
    return 1.0 - overlap, overlap


def pose_match_score(record_a, record_b):
    kp_dist, shared = keypoint_distance(record_a, record_b)
    if not math.isfinite(kp_dist):
        return None
    vis_penalty, vis_overlap = visibility_overlap_penalty(record_a, record_b)
    score = (0.80 * kp_dist) + (0.20 * vis_penalty)
    return {
        "pose_score": round(float(score), 6),
        "normalized_keypoint_distance": round(float(kp_dist), 6),
        "visibility_penalty": round(float(vis_penalty), 6),
        "visibility_overlap": round(float(vis_overlap), 6),
        "shared_keypoints": int(shared),
    }


def find_similar_poses(relative_path, top_k=8, same_pool=True):
    anchor = RECORDS_BY_PATH[relative_path]
    matches = []
    pool_field = "current_candidate_pool_key" if anchor.get("current_pipeline_pass") else "candidate_pool_key"
    for candidate in ok_records:
        if candidate["relative_path"] == relative_path:
            continue
        if same_pool and candidate.get(pool_field) != anchor.get(pool_field):
            continue
        score = pose_match_score(anchor, candidate)
        if score is None:
            continue
        matches.append(
            {
                "relative_path": candidate["relative_path"],
                "candidate_pool_key": candidate["candidate_pool_key"],
                "current_candidate_pool_key": candidate.get("current_candidate_pool_key"),
                "replace_mode": candidate["replace_mode"],
                "coverage_class": candidate["coverage_class"],
                **score,
            }
        )
    return sorted(matches, key=lambda item: item["pose_score"])[:top_k]


def show_match_gallery(relative_path, top_k=6):
    anchor = RECORDS_BY_PATH[relative_path]
    matches = find_similar_poses(relative_path, top_k=top_k, same_pool=True)
    print("Anchor:")
    print_record_summary(relative_path)
    print("\nTop matches:")
    for item in matches:
        print(
            f" - {item['relative_path']} | score={item['pose_score']} | "
            f"{item['coverage_class']} | current_pool={item.get('current_candidate_pool_key')}"
        )
    total = len(matches) + 1
    cols = min(4, total)
    rows = int(math.ceil(total / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))
    anchor_pool = anchor["current_candidate_pool_key"] if anchor.get("current_pipeline_pass") else anchor["candidate_pool_key"]
    all_items = [{"relative_path": relative_path, "title": f"ANCHOR\n{anchor_pool}"}]
    all_items.extend(
        [{"relative_path": item["relative_path"], "title": f"{item['pose_score']}\n{item['coverage_class']}"} for item in matches]
    )
    for idx, item in enumerate(all_items, start=1):
        plt.subplot(rows, cols, idx)
        plt.imshow(read_rgb(PROJECT_ROOT / item["relative_path"]))
        plt.title(item["title"], fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    return matches


def choose_demo_record(prefer_mode="full_body"):
    candidates = [record for record in CURRENT_STAGE_RECORDS if record["replace_mode"] == prefer_mode]
    if not candidates:
        candidates = CURRENT_STAGE_RECORDS or ok_records
    return min(
        candidates,
        key=lambda record: (-record["visible_keypoints"], -record["detection_conf"], record.get("bbox_area_ratio", 1.0)),
    )


def plot_pose_embedding(max_points=2500, color_by="replace_mode", seed=7):
    rng = np.random.default_rng(seed)
    candidates = [record for record in ok_records if record.get("normalized_keypoints") is not None]
    if len(candidates) > max_points:
        indices = rng.choice(len(candidates), size=max_points, replace=False)
        candidates = [candidates[int(i)] for i in indices]
    matrix = []
    labels = []
    for record in candidates:
        arr = normalized_array_from_record(record)
        matrix.append(np.nan_to_num(arr, nan=0.0).reshape(-1))
        labels.append(record.get(color_by, "unknown"))
    X = np.array(matrix, dtype=np.float32)
    X = X - X.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(X, full_matrices=False)
    coords = X @ vt[:2].T
    unique_labels = list(dict.fromkeys(labels))
    cmap = plt.get_cmap("tab10")
    plt.figure(figsize=(9, 7))
    for idx, label in enumerate(unique_labels):
        mask = np.array([item == label for item in labels], dtype=bool)
        pts = coords[mask]
        plt.scatter(pts[:, 0], pts[:, 1], s=14, alpha=0.55, label=str(label), color=cmap(idx % 10))
    plt.title(f"Pose embedding colored by {color_by}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend(markerscale=1.5, fontsize=8)
    plt.tight_layout()
    plt.show()


def top_label_with_other(labels, top_k=10):
    counts = Counter(labels)
    keep = {label for label, _ in counts.most_common(top_k)}
    return [label if label in keep else "other" for label in labels]


def plot_replaceable_tsne(max_points=2500, color_by="current_candidate_pool_key", top_k_labels=12, seed=7):
    from sklearn.manifold import TSNE

    rng = np.random.default_rng(seed)
    candidates = [record for record in CURRENT_STAGE_RECORDS if record.get("normalized_keypoints") is not None]
    if not candidates:
        print("No current-stage replaceable records are available.")
        return
    if len(candidates) > max_points:
        indices = rng.choice(len(candidates), size=max_points, replace=False)
        candidates = [candidates[int(i)] for i in indices]

    matrix = []
    labels = []
    for record in candidates:
        arr = normalized_array_from_record(record)
        matrix.append(np.nan_to_num(arr, nan=0.0).reshape(-1))
        labels.append(record.get(color_by, "unknown"))

    labels = top_label_with_other(labels, top_k=top_k_labels)
    X = np.array(matrix, dtype=np.float32)
    perplexity = max(8, min(35, len(candidates) // 12))
    perplexity = min(perplexity, max(5, len(candidates) - 1))
    coords = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed,
    ).fit_transform(X)

    unique_labels = list(dict.fromkeys(labels))
    cmap = plt.get_cmap("tab20")
    plt.figure(figsize=(11, 8))
    for idx, label in enumerate(unique_labels):
        mask = np.array([item == label for item in labels], dtype=bool)
        pts = coords[mask]
        plt.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.68, label=str(label), color=cmap(idx % 20))
    plt.title(f"Replaceable-dog t-SNE colored by {color_by}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(markerscale=1.4, fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


## Global View: Which Images Are Actually Replaceable?

This section focuses on the **current project stage only**: side-profile, full-body dogs that passed the replacement gate.

It now gives you two complementary views:

1. `t-SNE` over replaceable dogs only, so pose neighborhoods are easier to see
2. exported image folders under `data/replacbale_group/`, where every replaceable group gets its own folder and every non-replaceable image is collected into one reject group


In [ ]:
DEMO_RECORD = choose_demo_record(prefer_mode="full_body")
DEMO_PATH = DEMO_RECORD["relative_path"]
DEMO_POOL = DEMO_RECORD["candidate_pool_key"]

print_record_summary(DEMO_PATH)
draw_pose_record(DEMO_PATH, show_labels=False, figsize=(7, 7))
show_group_samples(DEMO_POOL, group_field="candidate_pool_key", max_images=8, cols=4)
neighbor_matches = show_match_gallery(DEMO_PATH, top_k=7)
neighbor_matches[:5]


In [ ]:
plot_replaceable_tsne(max_points=2500, color_by="pose_geometry_label", top_k_labels=10)
plot_replaceable_tsne(max_points=2500, color_by="current_candidate_pool_key", top_k_labels=12)
